# Analysez les données de systèmes éducatifs

## Exercice partie 1

### Etape 1 - Chargez les données dans votre Notebook

In [3]:
import pandas as pd

# Permet d'afficher toutes les colonnes du dataframe
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [4]:
imported_countries = pd.read_csv('./data/EdStatsCountry.csv')
imported_country_series = pd.read_csv('./data/EdStatsCountry-Series.csv')
imported_data = pd.read_csv('./data/EdStatsData.csv')
imported_footnote = pd.read_csv('./data/EdStatsFootNote.csv')
imported_series = pd.read_csv('./data/EdStatsSeries.csv')

### Etape 2 - Collectez les informations basiques sur chaque jeu de données

In [ ]:
# Fonction d'analyse des jeux de données
def analyze(file_name, data, categorial_columns):

    print(f"Le fichier {file_name} contient {data.shape[0]} lignes pour {data.shape[1]} colonnes. Il y à {data.duplicated().sum()} lignes dupliquée.")
    print("\n")

    for column in data.columns:
        print(column)
        nb_cell_empty = data[column].isnull().sum()
        nb_cell_filled = data[column].isnull().sum()
        print(f"Colonne {column} : {nb_cell_empty} cellules vides sur {data.shape[0]}, Soit {round((100 * nb_cell_empty) / data.shape[0])}% cellules vides.")

        # On utilise describe pour les colonnes numériques
        if data[column].dtype == 'int64' or data[column].dtype == 'float64':
            print(data[column].describe())

        # On affiche le nombre d'occurence par groupe pour les colonnes catégorielles
        if column in categorial_columns:
            print(data.groupby(column).size())

        print("\n")

#### **Analyse du fichier EdStatsCountry**

Chaque ligne du fichiers contient des informations sur un pays. Par exemple ci-dessus les informations du pays FRANCE


In [ ]:
# Exemple d'une ligne du jeu de donnée
imported_countries.loc[imported_countries['Country Code'] == 'FRA']

In [ ]:
# Supression d'une colonne inconnue
imported_countries = imported_countries.drop('Unnamed: 31', axis=1)

categorial_cols = ['Region', 'Income Group', 'National accounts base year', 'SNA price valuation', 'Lending category', 'Other groups', 'System of National Accounts', 'PPP survey year', 'External debt Reporting status', 'System of trade', 'Government Accounting concept', 'IMF data dissemination standard', 'Latest population census']

# Analyse du fichier
analyze('EdStatsCountry', imported_countries, categorial_cols)

# On renomme cette colonne pour simplifier les jointures + tard
imported_countries.rename(columns={'Country Code' : 'CountryCode'}, inplace=True)

#### **Analyse du fichier EdStatsCountry-Series**

Chaque ligne du fichier contient des informations complémentaires sur les données.

C'est aussi une table de correspondance entre le fichier Pays et le fichier Stats

La clé CountryCode correspond a la clé Country Code dans le fichier EdStatsCountry, et la clé SeriesCode correspond à la clé Indicator Code dans le fichier EdStatsData


In [ ]:
imported_country_series.loc[imported_country_series['CountryCode'] == 'FRA']

In [ ]:
# Supression d'une colonne
imported_country_series = imported_country_series.drop('Unnamed: 3', axis=1)

analyze('EdStatsCountrySeries', imported_country_series, [])

#### **Anayse du fichier EdStatsData**

Chaque ligne du fichier contient des statistiques par année

In [ ]:
imported_data.head(2)

In [ ]:
imported_data.drop('Unnamed: 69', axis=1)

analyze('EdStatsData', imported_data, [])

# On renomme la colonne maintenant pour simplifier les jointures plus tard
imported_data.rename(columns={'Indicator Code' : 'SeriesCode'}, inplace=True)

#### **Analyse du fichier EdStatsFootNote**

Chaque ligne du fichier contient du détail pour certaines lignes de EdStatsData

La clé SeriesCode correspond à la clé Indicator Code dans le fichier EdStatsData

In [ ]:
imported_footnote.head(2)

In [ ]:
stats_footnote = imported_footnote.drop('Unnamed: 4', axis=1)

analyze('EdStatsFootNote', imported_footnote, [])

#### **Analyse du fichier EdStatsSeries**

Chaque ligne du fichier contient des explication concernant le type de donnée enregistré dans chaque cellule par année.

La clé Series Code correspond à la clé Indicator Code dans le fichier EdStatsData

In [ ]:
imported_series.head(2)

In [ ]:
# Supression d'une colonne inconnue
imported_series = imported_series.drop('Unnamed: 20', axis=1)

analyze('EdStatsSeries', imported_series, [])

imported_series.rename(columns={'Series Code' : 'SeriesCode'}, inplace=True)

### Etape 3 - Réalisez votre premier nettoyage

#### Supression des faux pays

In [ ]:
countries_cleaned = imported_countries[ ~ imported_countries['Region'].isnull()]

Ma méthode naïve a été de téléchargé la liste des country code disponible sur kaggle ici : https://www.kaggle.com/datasets/wbdill/country-codes-iso-3166?resource=download

Et j'ai mergé le fichier EdStatsCountry en inner pour ne garder que les codes des pays qui sont présent dans les deux fichiers.

In [ ]:
import pandas as pd
countries_iso3166b = pd.read_csv('./data/countries_iso3166b.csv', sep=',', na_values=[''], quotechar='"', encoding='ISO-8859-1')
filtered_countries = imported_countries.merge(countries_iso3166b, left_on='Country Code', right_on='iso3', how='inner')
filtered_countries.head()

Mais j'ai remarqué après ques les faux pays sont ceux du jeu de donnée qui n'ont pas de région associée. On peut filtrer les pays de cette manière.

In [ ]:
countries_cleaned = imported_countries[ ~ imported_countries['Region'].isnull()]

#### Supression des faux pays dans les autres df

##### Méthode 1

En stockant les faux pays dans une liste qui sera utilisée pour le filtrage des différents dataframes.


In [ ]:
# On enregistre les pays a supprimer
countries_to_remove = imported_countries[imported_countries['Region'].isnull()]
print(f'Country a supprimer : {countries_to_remove.shape[0]}')
print(f'Country restant : {imported_countries.shape[0] - countries_to_remove.shape[0]}')

# On supprime les country series dont le countryCode est dans la liste des pays à supprimer
print(f'Country series avant : {imported_country_series.shape}')
method_one_country_series_filtered = imported_country_series[ ~ imported_country_series['CountryCode'].isin(countries_to_remove['CountryCode'])]
print(f'Country series après : {method_one_country_series_filtered.shape}')

# On supprime les stats data qui on un Indicator Code qui n'est pas dans les SeriesCode des country_series_filtered
print(f'Stats data avant : {imported_data.shape}')
method_one_stats_filtered = imported_data[imported_data['SeriesCode'].isin(method_one_country_series_filtered['SeriesCode'])]
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On supprime les footnote qui on un SeriesCode qui n'est pas dans les SeriesCode des country_series_filtered
print(f'Footnote avant : {imported_footnote.shape}')
method_one_footnote_filtered = imported_footnote[imported_footnote['SeriesCode'].isin(method_one_country_series_filtered['SeriesCode'])]
print(f'Footnote après : {method_one_footnote_filtered.shape}')


##### Méthode 2

En utilisant un inner join entre les pays du dataframe Country nettoyé, et les autres dataframes.

In [ ]:
# On joint le df country au df country_series sur la colonne CountryCode en inner pour s'assurer d'avoir une jointure sans faux pays
print(f'Country series avant : {imported_country_series.shape}')
method_two_country_series_fitered = imported_country_series.merge(countries_cleaned, how='inner')
print(f'Country series avant : {method_two_country_series_fitered.shape}')

# On joint a le df method_two_country_series_fitered au df stats en inner pour s'assurer de ne garder que les stats qui ont un SeriesCode qui est présent dans les SeriesCode de notre df method_two_country_series_fitered
print(f'Stats data avant : {imported_data.shape}')
method_one_stats_filtered = imported_data.merge(method_two_country_series_fitered, how='inner')
print(f'Stats data après : {method_one_stats_filtered.shape}')

# On joint
print(f'Footnote avant : {imported_footnote.shape}')
method_one_footnote_filtered = imported_footnote.merge(method_two_country_series_fitered, how='inner')
print(f'Footnote après : {method_one_footnote_filtered.shape}')